In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

import sys
sys.path.append("..")
from src.preprocessing import df_to_densities
from src.forecasting import cv


import warnings
from scipy.integrate import IntegrationWarning

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=IntegrationWarning)

Steps: <br>
1. Use cross-validation to select the best parameters for in-sample KDE <br>
2. Use cross-validation to select the number of dimensions for the dFPC using the resulting parameters for KDE in 1.

In [2]:
# Data
data_path = "../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df = pd.read_excel(returns_path, index_col="time")

# 1. Selecting KDE parameters

In [15]:
# Parameters to cross-validate
density_param_grid_0 = [
    {'kernel': ['gaussian', 'epanechnikov'],
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}, 
    {'kernel': ['t_student'], 
        'df': range(2,6),
        'bandwidth': ['silverman', 'scott'],#, 'cv'],
        'adaptive': [True, False]}]

density_param_grid = list(ParameterGrid(density_param_grid_0))
len(density_param_grid)

24

In [24]:
params_dict = {}
records = []

normalize_options = [True, False]
total_params = len(density_param_grid) * len(normalize_options)
i=1
for normalize in normalize_options:
    for params in density_param_grid:
        print(f"({i}/{total_params}) | Normalize = {normalize} | {params}")
        i += 1

        df_support, df_densities = df_to_densities(
                                        df, 
                                        params, 
                                        normalize_densities=normalize,
                                        verbose=False)
        try:
            cv_measures = cv(df_densities, df_support, initial_window=100)
        except Exception as e:
            print(f"\t ERROR: {e}")
            continue
        for m in cv_measures:
            record = {
                # density
                "normalized_density": normalize,
                
                # parameters
                "kernel": params["kernel"],
                "bandwidth": params["bandwidth"],
                "adaptive": params["adaptive"],

                # CV info
                "fold": m["fold"]+1,
                "method": m["method"],

                # metrics
                "KLD": float(m["KLD"]),
                "JSD": float(m["JSD"]),
                "L_1": float(m["L_1"]),
                "L_2": float(m["L_2"]),
                "L_INFTY": float(m["L_INFTY"]),
            }

            records.append(record)
            
            


results_df = pd.DataFrame(records)

(1/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'gaussian'}
>>> cv 1/149
	 ERROR: Array must not contain infs or NaNs
(2/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'silverman', 'kernel': 'epanechnikov'}
>>> cv 1/149
	 ERROR: Array must not contain infs or NaNs
(3/48) | Normalize = True | {'adaptive': True, 'bandwidth': 'scott', 'kernel': 'gaussian'}
>>> cv 1/149
>>> cv 2/149
>>> cv 3/149
>>> cv 4/149
>>> cv 5/149
>>> cv 6/149
>>> cv 7/149
>>> cv 8/149
>>> cv 9/149
>>> cv 10/149
>>> cv 11/149
>>> cv 12/149
>>> cv 13/149
>>> cv 14/149
>>> cv 15/149
>>> cv 16/149
>>> cv 17/149
>>> cv 18/149
>>> cv 19/149
>>> cv 20/149
>>> cv 21/149
>>> cv 22/149
>>> cv 23/149
>>> cv 24/149
>>> cv 25/149
>>> cv 26/149
>>> cv 27/149
>>> cv 28/149
>>> cv 29/149
>>> cv 30/149
>>> cv 31/149
>>> cv 32/149
>>> cv 33/149
>>> cv 34/149
>>> cv 35/149
>>> cv 36/149
>>> cv 37/149
>>> cv 38/149
>>> cv 39/149
>>> cv 40/149
>>> cv 41/149
>>> cv 42/149
>>> cv 43/149
>>> cv

KeyboardInterrupt: 

In [11]:
results_df_1

,density_model,kernel,bandwidth,adaptive,fold,method,KLD,JSD,L_1,L_2,L_INFTY
0,f_hat,gaussian,scott,True,0,KLE,0.023020,0.001472,12495.481608,664.607950,62.996451
1,f_hat,epanechnikov,scott,True,0,KLE,0.119637,0.007674,26831.110331,1764.932561,245.509173
2,f_hat,gaussian,cv,True,0,KLE,0.004913,0.000037,56.828245,35.789149,25.336447
3,f_hat,epanechnikov,cv,True,0,KLE,0.004843,0.000035,60.999210,35.313871,25.031146
4,f_hat,epanechnikov,scott,False,0,KLE,0.245792,0.007179,24939.456909,1570.561596,203.728073
5,f_hat,gaussian,cv,False,0,KLE,0.004913,0.000037,56.868220,35.789134,25.336442
6,f_hat,epanechnikov,cv,False,0,KLE,0.004843,0.000035,61.040793,35.313857,25.031141
7,f_hat,t_student,silverman,True,0,KLE,0.048028,0.003862,15565.940821,1115.032528,172.394931
8,f_hat,t_student,silverman,True,0,KLE,0.059206,0.004475,21296.850925,1354.735295,185.592675
9,f_hat,t_student,silverman,True,0,KLE,0.066887,0.004790,21951.310880,1400.994304,192.266120


In [ ]:
results_df.to_excel("../data/processed/cv_density_estimation_v2.xlsx", index=False)

In [ ]:
dimensions = np.arange(2,9)

array([2, 3, 4, 5, 6, 7, 8])